In [1]:
import os
import requests
import xml.etree.ElementTree as ET
from datetime import datetime

/opt/anaconda3/envs/AgentsIA/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
from random import randint
from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatClient

In [3]:
#Estado de memoria para detectar cambios
_estado_anterior = {
    "fecha_generacion": None,
    "total_individuos": None,
    "ultima_consulta": None,
}

In [4]:
URL_XML = "https://scsanctions.un.org/resources/xml/sp/consolidated.xml"

In [5]:
#Funciones que usaré para la extracción de información
def get_text(elemento, tag, default="nd"):
    nodo = elemento.find(tag)
    if nodo is not None and nodo.text:
        return nodo.text.strip()
    return default

def get_multiple_values(elemento, tag_padre, tag_hijo="VALUE"):
    valores = []
    for bloque in elemento.findall(tag_padre):
        val = get_text(bloque, tag_hijo)
        if val != "nd":
            valores.append(val)
    return " | ".join(valores) if valores else "nd"

def descargar_xml():
    """Descarga y parsea el XML de la ONU. Retorna el root o None si falla."""
    try:
        response = requests.get(URL_XML, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
        response.raise_for_status()
        return ET.fromstring(response.content)
    except Exception as e:
        return None

In [6]:
#Inicio de creación de agente
@tool
def verificar_actualizacion_lista() -> str:
    """
    Verifica si hubo cambios en el listado consolidado de sanciones de la ONU
    comparando la fecha de generación y el total de individuos con la última
    consulta registrada.

    Returns:
        str: Resumen del estado actual del listado, indicando si hubo cambios.
    """

    global _estado_anterior
    root = descargar_xml()

    if root is None:
        return "No se pudo descargar el XML de la ONU."

    fecha_generacion = root.get("dateGenerated", "nd")

    total_individuos = len(list(root.iter("INDIVIDUAL")))
    total_entidades  = len(list(root.iter("ENTITY")))

    ahora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    cambios = []
    if _estado_anterior["fecha_generacion"] is None:
        estado_msg = "Primera consulta registrada en esta sesión."
    else:
        if _estado_anterior["fecha_generacion"] != fecha_generacion:
            cambios.append(
                f"Fecha de generación cambió: "
                f"{_estado_anterior['fecha_generacion']} → {fecha_generacion}"
            )
        if _estado_anterior["total_individuos"] != total_individuos:
            diff = total_individuos - _estado_anterior["total_individuos"]
            signo = "+" if diff > 0 else ""
            cambios.append(
                f"Total de individuos cambió: "
                f"{_estado_anterior['total_individuos']} → {total_individuos} ({signo}{diff})"
            )
        estado_msg = "Sin cambios desde la última consulta." if not cambios else "\n".join(cambios)

        # Actualizar estado en memoria
    _estado_anterior["fecha_generacion"] = fecha_generacion
    _estado_anterior["total_individuos"] = total_individuos
    _estado_anterior["ultima_consulta"]  = ahora

    return (
        f"Listado Consolidado de Sanciones - ONU\n"
        f"{'─'*45}\n"
        f"Fecha de generación del XML : {fecha_generacion}\n"
        f"Total individuos sancionados: {total_individuos}\n"
        f"Total entidades sancionadas : {total_entidades}\n"
        f"Consultado en               : {ahora}\n"
        f"{'─'*45}\n"
        f"{estado_msg}\n"
        f"Fuente: https://main.un.org/securitycouncil/es/content/un-sc-consolidated-list"
    )


In [7]:
openai_chat_client = OpenAIChatClient(
   #base_url=os.environ.get("GITHUB_ENDPOINT"),
    #api_key=os.environ.get("GITHUB_TOKEN"),
    #model_id=os.environ.get("GITHUB_MODEL_ID")

)

In [8]:
agent = Agent(
    client=openai_chat_client,
    instructions="""Eres un asistente especializado en el monitoreo del Listado
    Consolidado de Sanciones del Consejo de Seguridad de la ONU.
    Puedes:
    - Verificar si hubo cambios o actualizaciones en el listado
    - Informar la fecha de la última revisión/generación del listado
    Siempre responde en español y sé conciso pero informativo.""",
    tools=[verificar_actualizacion_lista]  # Our random destination tool function
)

In [12]:
async def main():
    print("Agente de Monitoreo - Listado ONU")
    print("Preguntas de ejemplo:")
    print("'¿Hubo cambios en el listado?'")
    print("'¿Cuándo fue la última actualización?'")
    print("('salir' para terminar)\n")

    while True:
        user_input = input("# Tú: ").strip()

        if not user_input:
            continue
        if user_input.lower() in ["salir", "exit", "quit", "bye"]:
            print("¡Hasta luego!")
            break

        response = await agent.run(messages=user_input)
        last_message = response.messages[-1]


        if hasattr(last_message, "contents") and last_message.contents:
            text_content = last_message.contents[0].text
        elif hasattr(last_message, "content"):
            text_content = last_message.content
        else:
            text_content = str(last_message)

        role = getattr(last_message, "role", "Agente")
        print(f"\n# {role}: {text_content}\n")

await main()

Agente de Monitoreo - Listado ONU
Preguntas de ejemplo:
'¿Hubo cambios en el listado?'
'¿Cuándo fue la última actualización?'
('salir' para terminar)


# assistant: Parece que mencionaste "Lina". ¿Puedes especificar qué información necesitas sobre ella o si te refieres a algo relacionado con el Listado Consolidado de Sanciones de la ONU?


# assistant: No ha habido cambios en el listado consolidado de sanciones desde la última consulta. La fecha de generación del listado es del 17 de marzo de 2026, y el total de individuos sancionados es 730. Puedes consultar más detalles en la [fuente oficial](https://main.un.org/securitycouncil/es/content/un-sc-consolidated-list).

¡Hasta luego!
